# Introduction

In a different Notebook (see reference [1]), I visualized the map with locations (and details about) all quadruple and triple Grandmasters. The Notebooks used an idea from reference [2] and visualization solution for rich html popups in folium from reference [3].  

Let's see now the distribution of all Kaggle Grandmasters.

# Data and Utils

In [ ]:
import pandas as pd
import datetime
from IPython.display import Markdown
from IPython.display import Image
import folium
import numpy as np
from folium.plugins import MarkerCluster
from folium import Marker
n_users = pd.read_csv("../input/meta-kaggle/Users.csv")
master_achievements_df = pd.read_csv("../input/meta-kagglemaster-achievements-snapshot/MasterAchievements.csv")
master_achievements_df['GM_tier_count'] = master_achievements_df[master_achievements_df.astype(str) == 'grandmaster'].count(axis=1)
# select only rows with at least 1 GM tier
sel_gm_df = master_achievements_df[master_achievements_df['GM_tier_count'] > 0]
master_profiles_df = pd.read_csv("../input/meta-kagglemaster-achievements-snapshot/MasterProfiles.csv")
# merge profiles with selected users (GMs)
gm_profiles_df = sel_gm_df.merge(master_profiles_df)
# eliminate the GMs without location
gm_df = gm_profiles_df.loc[(~gm_profiles_df.Latitude.isna()) & (~gm_profiles_df.Longitude.isna())]

In [ ]:
def popup_html(row):

    user_name = row['UserName']
    user_avatar = row['Avatar']
    user_address = row['Location']
    user_country = row['Country']
    user_co = row['Competitions']
    user_da = row['Datasets']
    user_no = row['Notebooks']
    user_di = row['Discussion']
    highest_rank = row['HighestRank']
    metadata = row['Metadata']
    title_type = row['GM_tier_count']
    if title_type == 1:
        title = "Grandmaster"
    elif title_type == 2:
        title = "Double Grandmaster"
    elif title_type == 3:
        title = "Triple Grandmaster"
    else:
        title = "Quadruple Grandmaster"

    left_col_color = "#4166AA"
    right_col_color = "#f2f0d3"
    
    html = """<!DOCTYPE html>
<html>
<head>
<center><img src=\"""" + user_avatar + """\" alt="logo" width=100 height=100 ></center>
<h4 style="margin-bottom:10"; width="200px">{}</h4>""".format(user_name) + """
<h5 style="margin-bottom:10"; width="200px">{}</h5>""".format(title) + """
</head>
    <table style="height: 126px; width: 350px;">
<tbody>

<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Highest Rank</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(highest_rank) + """
</tr>

<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Competitions Tier</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(user_co) + """
</tr>
<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Datasets Tier</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(user_da) + """
</tr>
<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Notebooks Tier</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(user_no) + """
</tr>

<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Discussions Tier</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(user_di) + """
</tr>

<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Metadata</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(metadata) + """
</tr>

<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Address</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(user_address) + """
</tr>

<tr>
<td style="background-color: """+ left_col_color +""";"><span style="color: #ffffff;">Country</span></td>
<td style="width: 150px;background-color: """+ right_col_color +""";">{}</td>""".format(user_country) + """
</tr>

</tbody>
</table>
</html>
"""
    return html

# All Kaggle Grandmasters Map

In [ ]:
def show_map(grandmaster_df):
    gm_map = folium.Map(location=[0, 0], zoom_start=2)
    width = 200
    height = 500
    gm_popups, gm_locations = [], []

    for idx, row in grandmaster_df.iterrows():

        gm_locations.append([row['Latitude'], row['Longitude']])
        html = popup_html(row)
        popup = folium.Popup(folium.Html(html, script=True), max_width=500)
        gm_popups.append(popup)

    gm = folium.FeatureGroup(name='Grandmasters')
    gm.add_child(MarkerCluster(locations=gm_locations, popups=gm_popups))
    gm_map.add_child(gm)
    return gm_map

In [ ]:
gm_map = show_map(gm_df)
gm_map

# Competition Grandmasters Map

In [ ]:
cgm_df = gm_df.loc[gm_df.Competitions=="grandmaster"]
gm_map = show_map(cgm_df)
gm_map

# Datasets Grandmasters Map

In [ ]:
dgm_df = gm_df.loc[gm_df.Datasets=="grandmaster"]
gm_map = show_map(dgm_df)
gm_map

# Notebooks Grandmasters Map

In [ ]:
ngm_df = gm_df.loc[gm_df.Notebooks=="grandmaster"]
gm_map = show_map(ngm_df)
gm_map

# Discussion Grandmaster Map

In [ ]:
digm_df = gm_df.loc[gm_df.Discussion=="grandmaster"]
gm_map = show_map(digm_df)
gm_map

# Conclusions

On overall Kaggle Grandmasters distribution, Europe is leading, followed by North America, Japan, India and China.  
For Competitions, the top is dominated by Europe, North America, Japan and China.  
For Datasets, the top is India, North America, Europe and Middle East.  
For Notebooks, the top is India, Eastern Europe, Western Europe and North America.  
For Discussion, the top is Europe, North America and India.  

# References  

[1] Kaggle Quadruple and Triple Grandmasters Locations, https://www.kaggle.com/code/gpreda/kaggle-quadruple-and-triple-grandmasters-locations  
[2] Carl McBride Ellis, Kaggle in Numbers, https://www.kaggle.com/code/carlmcbrideellis/kaggle-in-numbers  
[3] Use HTML in Folium Maps: A Comprehensive Guide for Data Scientists, My Data Talk, Towards Data Science, https://towardsdatascience.com/use-html-in-folium-maps-a-comprehensive-guide-for-data-scientists-3af10baf9190  
